# S01a — Découvrir PySpark sur NYC Taxi

1. Démarrer Spark et repérer la **Spark UI** (http://localhost:4040)
2. Lire un mois de courses et comprendre son schéma
3. Comprendre l'**évaluation paresseuse** : transformations vs actions
4. Joindre le référentiel des zones et agréger
5. Ramener un petit résultat en pandas pour un premier graphique

Prérequis : `bash scripts/telecharger_donnees.sh` a été lancé (fichiers dans `data/raw/`).

In [ ]:
import sys; sys.path.append("/home/jovyan/work")
from pyspark.sql import functions as F
from commun.spark_utils import creer_session, RAW, LAKE

# Ici Spark seul : pas encore de data lake ni de base de données
spark = creer_session("S01a", lac=False, delta=False, postgres=False)

## 1. Lire les données

In [ ]:
MOIS = "2025-01"
trips = spark.read.parquet(f"{RAW}/yellow_tripdata_{MOIS}.parquet")
trips.printSchema()

In [ ]:
# Une ACTION : Spark calcule vraiment (onglet Jobs de la Spark UI)
print(f"{trips.count():,} courses")
trips.limit(5).toPandas()

## 2. Évaluation paresseuse

Une **transformation** (`filter`, `select`, `withColumn`, `groupBy`…) ne calcule rien : Spark note le plan.
Une **action** (`count`, `show`, `toPandas`, `write`…) exécute tout le plan.
La cellule suivante est instantanée et aucun job n'apparaît dans la Spark UI.

In [ ]:
longues = (trips
           .filter(F.col("trip_distance") > 20)
           .select("tpep_pickup_datetime", "trip_distance", "total_amount"))
longues.explain()     # le plan, pas le résultat

In [ ]:
longues.count()      # maintenant seulement, Spark calcule

In [ ]:
longues.limit(10).toPandas()      # maintenant seulement, Spark calcule

## 3. Joindre le référentiel des zones

`PULocationID` / `DOLocationID` = zones de départ / d'arrivée (1 à 265).

In [ ]:
zones = spark.read.csv(f"{RAW}/taxi_zone_lookup.csv", header=True, inferSchema=True)
zones.show(5)

trips_z = trips.join(
    zones.select(F.col("LocationID").alias("PULocationID"),
                 F.col("Borough").alias("pu_borough"),
                 F.col("Zone").alias("pu_zone")),
    on="PULocationID", how="left")

## 4. Premiers agrégats

In [ ]:
par_heure = (trips_z
             .withColumn("heure", F.hour("tpep_pickup_datetime"))
             .groupBy("heure")
             .agg(F.count("*").alias("courses"),
                  F.round(F.sum("total_amount"), 0).alias("revenu_total"),
                  F.round(F.avg("tip_amount"), 2).alias("pourboire_moyen"))
             .orderBy("heure"))
par_heure.show(24)

In [ ]:
(trips_z.groupBy("pu_borough")
        .agg(F.count("*").alias("courses"), F.round(F.avg("total_amount"), 2).alias("prix_moyen"))
        .orderBy(F.desc("courses"))
        .show())

## 5. Qualité des données

Ces anomalies seront traitées en S02 (couche *silver*).

In [ ]:
import datetime as dt
a, m = map(int, MOIS.split("-"))
debut, fin = f"{MOIS}-01", dt.date(a + (m == 12), m % 12 + 1, 1).isoformat()

trips.select(
    F.sum((F.col("total_amount") < 0).cast("int")).alias("montant_negatif"),
    F.sum(((F.col("passenger_count") == 0) | F.col("passenger_count").isNull()).cast("int")).alias("passagers_0_ou_vide"),
    F.sum((F.col("trip_distance") > 100).cast("int")).alias("distance_sup_100_miles"),
    F.sum(((F.col("tpep_pickup_datetime") < F.lit(debut)) |
           (F.col("tpep_pickup_datetime") >= F.lit(fin))).cast("int")).alias("hors_du_mois"),
).show()

## 6. Du résultat Spark au graphique

On agrège dans Spark, puis on ramène **un petit résultat** en pandas. Jamais `trips.toPandas()` sur des millions de lignes.

In [ ]:
import matplotlib.pyplot as plt
pdf = par_heure.toPandas()
fig, ax = plt.subplots(figsize=(9, 4))
ax.bar(pdf["heure"], pdf["courses"])
ax.set_xlabel("Heure de prise en charge"); ax.set_ylabel("Nombre de courses")
ax.set_title(f"Courses de taxis jaunes par heure — {MOIS}")
ax.set_xticks(range(0, 24, 2))
plt.show()

## À retenir
- Spark calcule en parallèle ; la Spark UI montre jobs et tâches.
- Transformations = plan ; actions = exécution.
- On agrège dans Spark, on visualise un petit résultat.

**Exercice :** quelle zone de départ a le revenu moyen par course le plus élevé ? Est-ce un aéroport ?

In [ ]:
spark.stop()